In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch

symbol = 'BTCUSDT'
market_type = 'SPOT'
discretization = '15M'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 17
macd_buffer = 200
atr_buffer = 200
display_start_date_str = '2026-01-01 00:00' # дата, с которой вы хотите отображать '2026-03-25 14:30:00' -- формат даты и времени
display_start_dt = parser.parse(display_start_date_str)

# Определяем таймфрейм в секундах
discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

# Рассчитываем дату начала загрузки с буфером
load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

print(f"Loading data from: {load_start_dt} (buffer {mrc_buffer} candles)")
print(f"Displaying from: {display_start_date_str}")

mrc_df = fetch(market_type, symbol, discretization, load_start_dt)
df = mrc_df.iloc[mrc_buffer:].copy()

print(df.head())

In [ ]:
def analyze_target_candle_outcome(df, target_idx, t_type):
    """
    Анализирует результат движения цены после целевой свечи
    Возвращает также границы для заливки (fill_start_level, fill_end_level)
    """
    import math

    # Получаем данные целевой свечи
    target_data = df.loc[[target_idx]]
    high = target_data['high'].iloc[0]
    low = target_data['low'].iloc[0]

    # Определяем базовые точки для расчета уровней
    if t_type == 's2':  # лонг
        log_start = math.log(high)
        log_end = math.log(low)
        all_levels = [0, -0.618, 0.618, 1, 1.618, 2.618]
        sl_level = 2.618
        success_levels_down = [0.618, 1, 1.618]
    else:  # шорт
        log_start = math.log(low)
        log_end = math.log(high)
        all_levels = [0, -0.618, -1.618, -2.618, -3.618]
        sl_level = -3.618
        success_levels_down = [-0.618, -1.618, -2.618]

    log_range = log_end - log_start

    # Рассчитываем цены всех уровней
    level_prices = {}
    for level in all_levels:
        level_prices[level] = math.exp(log_start + log_range * level)

    # Получаем данные после целевой свечи
    df_after = df.loc[target_idx:].iloc[1:].copy()

    if len(df_after) == 0:
        return {'status': 'in_progress', 'level': None, 'timestamp': None, 'fill_start_level': None, 'fill_end_level': None, 'details': 'No data after TC'}

    # Отслеживаем текущий "глубокий" уровень, которого достигла цена
    current_deep_level = 0  # начинаем с уровня 0

    for idx, row in df_after.iterrows():
        current_high = row['high']
        current_low = row['low']

        # Проверяем, достигла ли цена нового более глубокого уровня
        for level in success_levels_down:
            if current_low <= level_prices[level] <= current_high:
                if (t_type == 's2' and level > current_deep_level) or (t_type != 's2' and level < current_deep_level):
                    current_deep_level = level

        # Проверяем, вернулась ли цена на уровень выше текущего глубокого
        if current_deep_level != 0:
            # Определяем уровень выше (предыдущий)
            if t_type == 's2':  # лонг
                if current_deep_level == 0.618:
                    upper_level = 0
                elif current_deep_level == 1:
                    upper_level = 0.618
                elif current_deep_level == 1.618:
                    upper_level = 1
                else:
                    upper_level = None
            else:  # шорт
                if current_deep_level == -0.618:
                    upper_level = 0
                elif current_deep_level == -1.618:
                    upper_level = -0.618
                elif current_deep_level == -2.618:
                    upper_level = -1.618
                else:
                    upper_level = None

            if upper_level is not None:
                upper_price = level_prices[upper_level]
                if current_low <= upper_price <= current_high:
                    # Расчет изменений
                    price_start = level_prices[current_deep_level]
                    price_end = level_prices[upper_level]
                    price_change_abs = price_end - price_start
                    price_change_pct = (price_change_abs / price_start) * 100

                    return {
                        'status': 'success',
                        'level': current_deep_level,
                        'fill_start_level': current_deep_level,
                        'fill_end_level': upper_level,
                        'timestamp': idx,
                        'price_start': price_start,
                        'price_end': price_end,
                        'price_change_abs': price_change_abs,
                        'price_change_pct': price_change_pct,
                        'details': f'Price reached {current_deep_level}, then returned to {upper_level}'
                    }

        # Проверка успеха при движении вверх для лонга (достижение -0.618)
        if t_type == 's2' and current_low <= level_prices[-0.618] <= current_high:
            price_start = level_prices[0]
            price_end = level_prices[-0.618]
            price_change_abs = price_end - price_start
            price_change_pct = (price_change_abs / price_start) * 100

            return {
                'status': 'success',
                'level': 0,
                'fill_start_level': 0,
                'fill_end_level': -0.618,
                'timestamp': idx,
                'price_start': price_start,
                'price_end': price_end,
                'price_change_abs': price_change_abs,
                'price_change_pct': price_change_pct,
                'details': 'Price reached -0.618'
            }

        # Проверка Stop Loss
        if current_low <= level_prices[sl_level] <= current_high:
            # Определяем последний достигнутый уровень перед стоп-лоссом
            last_level = current_deep_level if current_deep_level != 0 else 0
            price_start = level_prices[last_level]
            price_end = level_prices[sl_level]
            price_change_abs = price_end - price_start
            price_change_pct = (price_change_abs / price_start) * 100

            return {
                'status': 'failure',
                'level': sl_level,
                'fill_start_level': last_level,
                'fill_end_level': sl_level,
                'timestamp': idx,
                'price_start': price_start,
                'price_end': price_end,
                'price_change_abs': price_change_abs,
                'price_change_pct': price_change_pct,
                'details': f'Price reached Stop Loss {sl_level}'
            }

    return {'status': 'in_progress', 'level': None, 'timestamp': None, 'fill_start_level': None, 'fill_end_level': None, 'details': 'Analysis not completed'}

def collect_statistics(df, target_indices):
    """
    Собирает статистику по всем целевым свечам
    """
    stats = {
        'total': len(target_indices),
        'long': 0,
        'short': 0,
        'success': 0,
        'failure': 0,
        'in_progress': 0,
        'by_level': {},  # успех по уровням
        'stop_loss': 0,  # поражение по стоп-лоссу
        'long_success': 0,
        'long_failure': 0,
        'short_success': 0,
        'short_failure': 0,
        'by_level_long': {},
        'by_level_short': {}
    }

    for idx in target_indices:
        t_type = df.loc[idx, 'target_candle_type'] if 'target_candle_type' in df.columns else None

        if t_type == 's2':
            stats['long'] += 1
        elif t_type == 'r2':
            stats['short'] += 1

        # Анализируем результат
        outcome = analyze_target_candle_outcome(df, idx, t_type)

        if outcome['status'] == 'success':
            stats['success'] += 1
            level = outcome['level']

            # Общая статистика по уровням
            if level not in stats['by_level']:
                stats['by_level'][level] = 0
            stats['by_level'][level] += 1

            # Статистика по типу сигнала
            if t_type == 's2':
                stats['long_success'] += 1
                if level not in stats['by_level_long']:
                    stats['by_level_long'][level] = 0
                stats['by_level_long'][level] += 1
            else:
                stats['short_success'] += 1
                if level not in stats['by_level_short']:
                    stats['by_level_short'][level] = 0
                stats['by_level_short'][level] += 1

        elif outcome['status'] == 'failure':
            stats['failure'] += 1
            stats['stop_loss'] += 1

            if t_type == 's2':
                stats['long_failure'] += 1
            else:
                stats['short_failure'] += 1

        elif outcome['status'] == 'in_progress':
            stats['in_progress'] += 1

    return stats

def print_statistics(stats):
    """
    Prints statistics to the console
    """
    print("\n" + "="*60)
    print("📊 TARGET CANDLE STATISTICS")
    print("="*60)

    print(f"\n📌 GENERAL STATISTICS:")
    print(f"   Timeframe: {discretization}")
    print(f"   Start date: {display_start_date_str}")
    print(f"   Total target candles: {stats['total']}")
    print(f"   🟢 LONG signals: {stats['long']}")
    print(f"   🔴 SHORT signals: {stats['short']}")

    print(f"\n📈 RESULTS:")
    print(f"   ✅ SUCCESSFUL: {stats['success']} ({stats['success']/stats['total']*100:.1f}%)")
    print(f"   ❌ UNSUCCESSFUL: {stats['failure']} ({stats['failure']/stats['total']*100:.1f}%)")
    print(f"   🔄 IN PROGRESS: {stats['in_progress']} ({stats['in_progress']/stats['total']*100:.1f}%)")

    print(f"\n🎯 SUCCESS BY LEVEL (total):")
    for level in sorted(stats['by_level'].keys()):
        count = stats['by_level'][level]
        print(f"   Level {level}: {count} ({count/stats['success']*100:.1f}% of successful)")

    print(f"\n🛑 FAILURES:")
    print(f"   Stop Loss: {stats['stop_loss']}" if stats['failure'] > 0 else "   No failures")

    print(f"\n🟢 LONG SIGNALS:")
    print(f"   Total: {stats['long']}")
    print(f"   ✅ Success: {stats['long_success']} ({stats['long_success']/stats['long']*100:.1f}% of LONG)" if stats['long'] > 0 else "   No LONG signals")
    print(f"   ❌ Failure: {stats['long_failure']} ({stats['long_failure']/stats['long']*100:.1f}% of LONG)" if stats['long'] > 0 else "")

    if stats['by_level_long']:
        print(f"   LONG success levels:")
        for level in sorted(stats['by_level_long'].keys()):
            count = stats['by_level_long'][level]
            print(f"      Level {level}: {count}")

    print(f"\n🔴 SHORT SIGNALS:")
    print(f"   Total: {stats['short']}")
    print(f"   ✅ Success: {stats['short_success']} ({stats['short_success']/stats['short']*100:.1f}% of SHORT)" if stats['short'] > 0 else "   No SHORT signals")
    print(f"   ❌ Failure: {stats['short_failure']} ({stats['short_failure']/stats['short']*100:.1f}% of SHORT)" if stats['short'] > 0 else "")

    if stats['by_level_short']:
        print(f"   SHORT success levels:")
        for level in reversed(sorted(stats['by_level_short'].keys())):
            count = stats['by_level_short'][level]
            print(f"      Level {level}: {count}")

    print("\n" + "="*60)

def analyze_and_print_statistics(df):
    target_candle_indices = get_all_target_candle_indices(df)

    if not target_candle_indices:
        print("There are no target candles for analysis")
        return

    stats = collect_statistics(df, target_candle_indices)
    print_statistics(stats)

    return stats

In [ ]:
import ta
import numpy as np
import pandas as pd
#Process iteration over timeseries dataframe with sliding window approach
window_size = 30

def mrc_supersmoother(src, length):
    """
    Supersmoother filter by John Ehlers
    """
    a1 = np.exp(-1.414 * np.pi / length)
    b1 = 2 * a1 * np.cos(1.414 * np.pi / length)
    c3 = -a1 * a1
    c2 = b1
    c1 = 1 - c2 - c3

    ss = np.zeros_like(src)
    ss[:2] = src[:2]

    for i in range(2, len(src)):
        ss[i] = c1 * src[i] + c2 * ss[i-1] + c3 * ss[i-2]

    return ss

def mrc_sak_filter(filter_type, src, length):
    """
    Ehlers Swiss Army Knife filters
    """
    pi = np.pi
    cycle = 2 * pi / length

    c0, c1 = 1.0, 0.0
    b0, b1, b2 = 1.0, 0.0, 0.0
    a1, a2 = 0.0, 0.0
    alpha, beta, gamma = 0.0, 0.0, 0.0

    if filter_type == "Ehlers EMA":
        alpha = (np.cos(cycle) + np.sin(cycle) - 1) / np.cos(cycle)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "Gaussian":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "Butterworth":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha / 4
        b1, b2 = 2, 1
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "BandStop":
        beta = np.cos(cycle)
        gamma = 1 / np.cos(cycle * 2 * 0.1)  # delta = 0.1
        alpha = gamma - np.sqrt(gamma * gamma - 1)
        c0 = (1 + alpha) / 2
        b1 = -2 * beta
        b2 = 1
        a1 = beta * (1 + alpha)
        a2 = -alpha

    elif filter_type == "SMA":
        c1 = 1 / length
        b0 = 1 / length
        a1 = 1

    elif filter_type == "EMA":
        alpha = 2 / (length + 1)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "RMA":
        alpha = 1 / length
        b0 = alpha
        a1 = 1 - alpha

    output = np.zeros_like(src)
    output[:3] = src[:3]

    for i in range(3, len(src)):
        output[i] = (c0 * (b0 * src[i] +
                          b1 * (src[i-1] if i-1 >= 0 else 0) +
                          b2 * (src[i-2] if i-2 >= 0 else 0)) +
                    a1 * output[i-1] +
                    a2 * output[i-2] -
                    c1 * (src[i-length] if i-length >= 0 else 0))

    return output

def mrc_calculate(source_type='hlc3', filter_type='SuperSmoother', innermult=1.0, outermult=2.415):
    """
    Calculate Mean Reversion Channel

    Использует глобальные переменные:
    - mrc_df: DataFrame с данными (включая буфер)
    - df: DataFrame для отображения (без буфера)
    """
    global mrc_df, df

    # Calculate source price на mrc_df (с буфером)
    if source_type == 'hlc3':
        source = (mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 3
    elif source_type == 'ohlc4':
        source = (mrc_df['open'] + mrc_df['high'] + mrc_df['low'] + mrc_df['close']) / 4
    else:  # 'close'
        source = mrc_df['close']

    source = source.values

    # Calculate True Range на mrc_df
    tr = np.maximum(
        mrc_df['high'] - mrc_df['low'],
        np.maximum(
            abs(mrc_df['high'] - mrc_df['close'].shift(1)),
            abs(mrc_df['low'] - mrc_df['close'].shift(1))
        )
    ).fillna(0).values

    length = 200

    # Apply filters
    if filter_type == 'SuperSmoother':
        meanline = mrc_supersmoother(source, length)
        meanrange = mrc_supersmoother(tr, length)
    else:
        meanline = mrc_sak_filter(filter_type, source, length)
        meanrange = mrc_sak_filter(filter_type, tr, length)

    pi = np.pi
    mult = pi * innermult
    mult2 = pi * outermult

    # Calculate channels
    upband1 = meanline + (meanrange * mult)
    loband1 = meanline - (meanrange * mult)
    upband2 = meanline + (meanrange * mult2)
    loband2 = meanline - (meanrange * mult2)

    # Сохраняем результаты в mrc_df
    mrc_df['meanline'] = meanline
    mrc_df['meanrange'] = meanrange
    mrc_df['upband1'] = upband1
    mrc_df['loband1'] = loband1
    mrc_df['upband2'] = upband2
    mrc_df['loband2'] = loband2

    # Переносим результаты из mrc_df в df (видимый диапазон)
    df['meanline'] = mrc_df.loc[df.index, 'meanline']
    df['meanrange'] = mrc_df.loc[df.index, 'meanrange']
    df['upband1'] = mrc_df.loc[df.index, 'upband1']
    df['loband1'] = mrc_df.loc[df.index, 'loband1']
    df['upband2'] = mrc_df.loc[df.index, 'upband2']
    df['loband2'] = mrc_df.loc[df.index, 'loband2']

def stochastic_tradingview(high, low, close, periodK=14, smoothK=3, periodD=3):
    lowest_low = low.rolling(window=periodK).min()
    highest_high = high.rolling(window=periodK).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_k = raw_k.rolling(window=smoothK).mean()
    stoch_d = stoch_k.rolling(window=periodD).mean()

    return stoch_k, stoch_d

def find_target_candles(df_display, df_buffer, window=10):
    """
    Поиск целевых свечей по критериям:
    1. Касание линии R2 (upband2) или S2 (loband2) тенью или телом
       - Свеча пересекает линию (часть выше, часть ниже)
       - ИЛИ свеча полностью выше R2
       - ИЛИ свеча полностью ниже S2
    2. Объем выше среднего за 10 предыдущих свечей
    3. Тень больше средней тени за 10 предыдущих свечей

    Parameters:
    df_display : DataFrame с видимыми данными (после обрезки буфера)
    df_buffer : DataFrame с полными данными (включая буфер для расчета)
    window : период для скользящих средних
    """
    # Используем df_buffer как основу, добавляем df_display если нужно
    if df_display.index[0] in df_buffer.index:
        df_full = df_buffer.copy()
    else:
        df_full = pd.concat([df_buffer, df_display]).drop_duplicates().sort_index()

    # Расчет скользящих средних (только один раз)
    df_full['volume_ma'] = df_full['volume'].rolling(window=window, min_periods=1).mean()

    # Расчет теней
    df_full['upper_shadow'] = df_full['high'] - df_full[['open', 'close']].max(axis=1)
    df_full['lower_shadow'] = df_full[['open', 'close']].min(axis=1) - df_full['low']
    df_full['max_shadow'] = df_full[['upper_shadow', 'lower_shadow']].max(axis=1)
    df_full['shadow_ma'] = df_full['max_shadow'].shift(1).rolling(window=window, min_periods=1).mean()

    # Поиск целевых свечей только для видимого диапазона
    target_conditions = []

    for idx in df_display.index:
        i = df_full.index.get_loc(idx)
        if isinstance(i, (slice, np.ndarray)):
            i = i.start if isinstance(i, slice) else i[0]

        if i < window:
            target_conditions.append(False)
            continue

        high = df_full['high'].iloc[i]
        low = df_full['low'].iloc[i]
        r2 = df_full['upband2'].iloc[i]
        s2 = df_full['loband2'].iloc[i]

        touches_r2 = (high >= r2 and low <= r2) or (low > r2)
        touches_s2 = (high >= s2 and low <= s2) or (high < s2)
        touches_level = touches_r2 or touches_s2

        volume_above = df_full['volume'].iloc[i] > df_full['volume_ma'].iloc[i-1]
        shadow_above = df_full['max_shadow'].iloc[i] > df_full['shadow_ma'].iloc[i]

        target_conditions.append(touches_level and volume_above and shadow_above)

    df_display['is_target_candle'] = target_conditions

    return df_display

def get_target_candle_type(row):
    """Определяет тип целевой свечи"""
    if row['is_target_candle']:
        if (row['high'] >= row['upband2'] and row['low'] <= row['upband2']) or (row['low'] > row['upband2']):
            return 'r2'
        if (row['high'] >= row['loband2'] and row['low'] <= row['loband2']) or (row['high'] < row['loband2']):
            return 's2'

    return None

def get_all_target_candle_indices(df):
    """Находит все целевые свечи в датасете"""
    targets = df[df['is_target_candle']]

    if len(targets) == 0:
        print("Target candles not found")
        return []

    return targets.index.tolist()

# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# Функция для подготовки данных с буфером
def prepare_buffer_data(df_main, df_display, buffer_size):
    combined = pd.concat([df_main.iloc[-(buffer_size + len(df_display)):], df_display])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    return combined

# 1. RSI
rsi_calc_df = prepare_buffer_data(mrc_df, df, rsi_buffer)
df['rsi'] = ta.momentum.RSIIndicator(close=rsi_calc_df['close'], window=14).rsi().loc[df.index]

# 2. Stochastic
stoch_calc_df = prepare_buffer_data(mrc_df, df, stoch_buffer)
stoch_k_full, stoch_d_full = stochastic_tradingview(
    high=stoch_calc_df['high'],
    low=stoch_calc_df['low'],
    close=stoch_calc_df['close']
)
df['stoch_k'] = stoch_k_full.loc[df.index]
df['stoch_d'] = stoch_d_full.loc[df.index]

# 3. MACD
macd_calc_df = prepare_buffer_data(mrc_df, df, macd_buffer)
macd = ta.trend.MACD(
    close=macd_calc_df['close'],
    window_slow=26,
    window_fast=12,
    window_sign=9
)
df['macd_line'] = macd.macd().loc[df.index]
df['macd_signal'] = macd.macd_signal().loc[df.index]
df['macd_histogram'] = macd.macd_diff().loc[df.index]

# 4. ATR
atr_calc_df = prepare_buffer_data(mrc_df, df, atr_buffer)
atr_full = ta.volatility.AverageTrueRange(
    high=atr_calc_df['high'],
    low=atr_calc_df['low'],
    close=atr_calc_df['close'],
    window=14
).average_true_range()
df['atr'] = atr_full.loc[df.index]

for i in range(len(df) - window_size + 1):
    window = df.iloc[i:i + window_size]
    # process window
    print(f"{window.index[0]} - {window.index[-1]}")

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

is_log_scale_by_default=True
candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

def add_bars(col, name, color, row):
    fig_main.add_trace(
        go.Bar(
            x=df.index,
            y=df[col],
            name=name,
            marker=dict(color=color),
            width=(df.index[1] - df.index[0]).total_seconds() * 1000,
        ),
        row=row, col=1
)

def add_scatter(col, name, color, row, fill=None, fillcolor=None, width=2, dash=None):
    fig_main.add_trace(
        go.Scatter(
            x=df.index,
            y=df[col],
            name=name,
            line=dict(color=color, width=width, dash=dash),
            mode='lines',
            fill=fill,
            fillcolor=fillcolor
        ),
        row=row, col=1
    )

def add_target_candle_scatter(target_candle_type, position, position_multiplier, name, color, symbol_direction):
    signals = df[df['target_candle_type'] == target_candle_type]

    if len(signals) > 0:
        fig_main.add_trace(
            go.Scatter(
                x=signals.index,
                y=signals[position] * position_multiplier,
                name=name + " TC",
                mode='markers',
                marker=dict(color=color, size=10, symbol='triangle-' + symbol_direction)
            ),
            row=candlestick_row, col=1
        )

def add_over_zone(y0, y1, fillcolor, row):
    fig_main.add_hrect(
        y0=y0, y1=y1,
        line_width=0,
        fillcolor=fillcolor,
        opacity=0.2,
        row=row, col=1
    )

def add_central_line(row, y=50, line_dash="dot"):
    fig_main.add_hline(
        y=y,
        line_dash=line_dash,
        line_color="white",
        line_width=1,
        opacity=0.3,
        row=row, col=1
    )

def add_over_zones_and_a_central_line(row):
    add_over_zone(80, 100, "red", row)
    add_over_zone(0, 20, "green", row)
    add_central_line(row)

def add_stoch(speed, color):
    add_scatter('stoch_' + speed, "Stoch %" + speed.capitalize(), color, stoch_row)

def add_fibonacci_levels(fig, target_data, target_idx, df_subset, row=1, col=1, fill_start_level=None, fill_end_level=None, fill_status=None, t_type=None):
    """
    Добавляет уровни Фибоначчи с продолжением вправо до конца графика
    Уровни рассчитываются в логарифмическом масштабе
    Для лонга: 0 = максимум, 1 = минимум
    Для шорта: 0 = минимум, 1 = максимум (зеркально)
    """
    import math

    high = target_data['high'].iloc[0]
    low = target_data['low'].iloc[0]

    if t_type is None:
        t_type = target_data['target_candle_type'].iloc[0] if 'target_candle_type' in target_data.columns else None

    # Определяем базовые точки в зависимости от типа сигнала
    if t_type == 's2':
        log_start = math.log(high)
        log_end = math.log(low)
    else:
        log_start = math.log(low)
        log_end = math.log(high)

    log_range = log_end - log_start

    levels_with_colors = {
        0: 'yellow',
        0.618: 'yellow',
        1: 'yellow',
        1.618: 'orange',
        2.618: 'aqua',
        -0.618: 'violet',
        -1.618: 'lavender',
        -2.618: 'orange',
        -3.618: 'aqua'
    }

    # Расчет цен уровней
    prices = {}
    for level, color in levels_with_colors.items():
        prices[level] = math.exp(log_start + log_range * level)

    time_offset = df_subset.index[0] - (df_subset.index[1] - df_subset.index[0])
    end_date = df_subset.index[-1]

    # Добавляем заливку между уровнями (если указаны)
    if fill_start_level is not None and fill_end_level is not None:
        if fill_start_level in prices and fill_end_level in prices:
            y_lower = min(prices[fill_start_level], prices[fill_end_level])
            y_upper = max(prices[fill_start_level], prices[fill_end_level])

            # Определяем цвет заливки в зависимости от статуса
            if fill_status == 'failure':
                fill_color = 'red'
            else:
                if fill_start_level in levels_with_colors:
                    fill_color = levels_with_colors[fill_start_level]
                else:
                    fill_color = 'gray'

            # Добавляем заливку от целевой свечи до конца графика
            fig.add_trace(
                go.Scatter(
                    x=[target_idx, end_date, end_date, target_idx],
                    y=[y_lower, y_lower, y_upper, y_upper],
                    fill='toself',
                    mode='lines',
                    name=f"Fill {fill_start_level} → {fill_end_level}",
                    line=dict(width=0),
                    fillcolor=fill_color,
                    opacity=0.3,
                    showlegend=False
                ),
                row=row, col=col
            )

    # Добавляем линии уровней и текст
    for l, p in prices.items():
        fig.add_trace(
            go.Scatter(
                x=[target_idx, end_date],
                y=[p, p],
                mode='lines',
                name=f"Fib {l}: {p:.2f}",
                line=dict(color=levels_with_colors[l], width=1),
                showlegend=True
            ),
            row=row, col=col
        )
        fig.add_trace(
            go.Scatter(
                x=[time_offset],
                y=[p],
                mode='text',
                text=[f"{l} ({p:.2f})"],
                textposition="middle left",
                textfont=dict(size=12, color=levels_with_colors[l], family="Arial"),
                showlegend=False,
                hoverinfo='none'
            ),
            row=row, col=col
        )

    return fig

def plot_target_candle_with_fib(df, target_idx, future_bars):
    """Строит график для одной целевой свечи с анализом результата"""
    target_pos = df.index.get_loc(target_idx)

    # Для графика берем future_bars свечей
    end_idx = min(len(df) - 1, target_pos + future_bars)
    df_subset = df.iloc[target_pos:end_idx + 1].copy()

    fig_target_candle = make_subplots(rows=1, cols=1)

    fig_target_candle.add_trace(
        go.Candlestick(
            x=df_subset.index,
            open=df_subset["open"],
            high=df_subset["high"],
            low=df_subset["low"],
            close=df_subset["close"],
            name="OHLC",
            showlegend=True
        ),
        row=1, col=1
    )

    target_data = df_subset.iloc[0:1]
    t_type = target_data['target_candle_type'].iloc[0] if 'target_candle_type' in target_data.columns else None

    # Определяем параметры в зависимости от типа сигнала
    if t_type == 'r2':  # R2 = SHORT
        line_color = 'tomato'
        signal_symbol = "🔴"
        signal_text = "SHORT SIGNAL"
        sl_level = -3.618
    elif t_type == 's2':  # S2 = LONG
        line_color = 'lime'
        signal_symbol = "🟢"
        signal_text = "LONG SIGNAL"
        sl_level = 2.618
    else:
        line_color = 'gold'
        signal_symbol = "⚪"
        signal_text = "UNKNOWN"
        sl_level = None

    # Анализируем результат
    if t_type in ['s2', 'r2']:
        outcome = analyze_target_candle_outcome(df, target_idx, t_type)
    else:
        outcome = {'status': 'unknown', 'level': None, 'fill_start_level': None, 'fill_end_level': None, 'timestamp': None, 'details': ''}

    # Формируем название графика
    date_str = target_idx.strftime('%Y-%m-%d %H:%M') if hasattr(target_idx, 'strftime') else str(target_idx)

    if outcome['status'] == 'success':
        result_str = f"✅ SUCCESS (level: {outcome['level']})"
        result_time = outcome['timestamp'].strftime('%Y-%m-%d %H:%M') if hasattr(outcome['timestamp'], 'strftime') else str(outcome['timestamp'])

        # Добавляем информацию об изменении цены
        price_info = f" | 📊 {outcome['price_change_abs']:.2f} ({outcome['price_change_pct']:.2f}%)"

        title = f"{signal_symbol} {signal_text} | 🎯 TC: {date_str} | {result_str} | 📍 Result: {result_time}{price_info}"
    elif outcome['status'] == 'failure':
        result_str = f"❌ FAILURE (SL: {outcome['level']})"
        result_time = outcome['timestamp'].strftime('%Y-%m-%d %H:%M') if hasattr(outcome['timestamp'], 'strftime') else str(outcome['timestamp'])

        # Добавляем информацию об изменении цены
        price_info = f" | 📊 {outcome['price_change_abs']:.2f} ({outcome['price_change_pct']:.2f}%)"

        title = f"{signal_symbol} {signal_text} | 🎯 TC: {date_str} | {result_str} | 📍 Result: {result_time}{price_info}"
    elif outcome['status'] == 'in_progress':
        title = f"{signal_symbol} {signal_text} | 🎯 TC: {date_str} | 🔄 IN PROGRESS"
    else:
        title = f"{signal_symbol} {signal_text} | 🎯 TC: {date_str} | ⚪ UNCERTAIN"

    fig_target_candle.add_trace(
        go.Scatter(
            x=[target_idx, target_idx],
            y=[df_subset['low'].min() * 0.98, df_subset['high'].max() * 1.02],
            mode='lines',
            name="🎯 Target Candle",
            line=dict(color=line_color, width=0.5, dash='dash'),
            showlegend=True
        ),
        row=1, col=1
    )

    # Добавляем уровни Фибоначчи с заливкой
    fig_target_candle = add_fibonacci_levels(
        fig_target_candle, target_data, target_idx, df_subset, row=1, col=1,
        fill_start_level=outcome.get('fill_start_level'),
        fill_end_level=outcome.get('fill_end_level'),
        fill_status=outcome['status'],
        t_type=t_type
    )

    # Добавляем Stop Loss
    if sl_level is not None:
        import math

        high = target_data['high'].iloc[0]
        low = target_data['low'].iloc[0]

        if t_type == 's2':  # лонг
            log_start = math.log(high)
            log_end = math.log(low)
        else:  # шорт
            log_start = math.log(low)
            log_end = math.log(high)

        log_range = log_end - log_start
        sl_price = math.exp(log_start + log_range * sl_level)

        fig_target_candle.add_trace(
            go.Scatter(
                x=[target_idx, df_subset.index[-1]],
                y=[sl_price, sl_price],
                mode='lines',
                name=f"🛑 Stop Loss ({sl_level})",
                line=dict(color='crimson', width=2, dash='dash'),
                showlegend=True
            ),
            row=1, col=1
        )

        fig_target_candle.add_trace(
            go.Scatter(
                x=[target_idx],
                y=[sl_price],
                mode='text',
                text=[f"🛑"],
                textposition="middle right",
                textfont=dict(size=11, color='red', family="Arial"),
                showlegend=False,
                hoverinfo='none'
            ),
            row=1, col=1
        )

    # Если результат был достигнут, добавляем вертикальную линию в момент результата
    if outcome['status'] in ['success', 'failure'] and outcome['timestamp'] is not None:
        result_time = outcome['timestamp']
        if result_time in df_subset.index:
            fig_target_candle.add_vline(
                x=result_time,
                line_dash="dash",
                line_color='lime' if outcome['status'] == 'success' else 'red',
                line_width=0.4,
                opacity=1,
                row=1, col=1
            )

    fig_target_candle.update_layout(
        title=title,
        xaxis_rangeslider_visible=False,
        yaxis_type="log",
        yaxis_title="Price (log scale)",
        template="plotly_dark",
        height=1700,
        showlegend=True,
        hovermode='x unified'
    )

    return fig_target_candle

def plot_all_target_candles_and_save_htmls_to_disc(df, future_bars, output_dir="target_candles", clean_dir=True):
    """Сохраняет графики в HTML файлы, с опциональной очисткой папки"""
    import shutil
    import os

    target_candle_indices = get_all_target_candle_indices(df)

    if not target_candle_indices:
        print("No target candles to display")
        return

    # Clean the folder if needed
    if clean_dir and os.path.exists(output_dir):
        print(f"Cleaning folder '{output_dir}'...")
        shutil.rmtree(output_dir)
        print(f"Folder '{output_dir}' deleted")

    # Create folder for charts
    os.makedirs(output_dir, exist_ok=True)
    print(f"Folder '{output_dir}' created")

    for i, target_idx in enumerate(target_candle_indices):
        date_str = target_idx.strftime('%Y-%m-%d_%H-%M') if hasattr(target_idx, 'strftime') else str(target_idx)
        filename = f"{output_dir}/target_{i+1:03d}_{date_str}.html"
        fig = plot_target_candle_with_fib(df, target_idx, future_bars)
        fig.write_html(filename)

    print(f"\n✅ All {len(target_candle_indices)} charts saved to folder '{output_dir}'")
    print(f"   Open the files in your browser to view")

fig_main = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[7, 1, 3, 3, 3, 2],
    vertical_spacing=0.03
)

fig_main.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

mrc_calculate()
add_scatter('meanline', "MRC Mean", '#FFCD00', candlestick_row)
add_scatter('upband1', "MRC R1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('loband1', "MRC S1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('upband2', "MRC R2", 'red', candlestick_row, width=1)
add_scatter('loband2', "MRC S2", 'red', candlestick_row, width=1)

add_bars(
    "volume",
    "Volume",
    ["green" if c > o else "red" for o, c in zip(df["open"], df["close"])],
    volume_row)

add_scatter('rsi', "RSI", 'purple', rsi_row)
add_over_zones_and_a_central_line(rsi_row)

add_stoch('k', "lightblue")
add_stoch('d', "orange")
add_over_zones_and_a_central_line(stoch_row)

full_histogram = macd.macd_diff()
prev_histogram = full_histogram.shift(1).loc[df.index]
conditions = [
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] >= prev_histogram),
    (df['macd_histogram'] >= 0) & (df['macd_histogram'] < prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] <= prev_histogram),
    (df['macd_histogram'] < 0) & (df['macd_histogram'] > prev_histogram)
]
choices = ['green', 'lightgreen', 'red', 'lightcoral']
macd_colors = np.select(conditions, choices, default='rgba(128, 128, 128, 0.3)')
add_scatter("macd_line", "MACD Line", 'lightblue', macd_row)
add_scatter("macd_signal", "Signal Line", 'orange', macd_row)
add_bars("macd_histogram", "MACD Histogram", macd_colors, macd_row)
add_central_line(macd_row, line_dash="solid")

add_scatter('atr', "ATR (14)", 'orange', atr_row, fill='tozeroy', fillcolor='rgba(255, 165, 0, 0.1)')
add_central_line(atr_row, y=df['atr'].mean())

df = find_target_candles(df, mrc_df)
df['target_candle_type'] = df.apply(get_target_candle_type, axis=1)
add_target_candle_scatter('r2', 'high', 1.005, "🔴 Short", 'red', "down")
add_target_candle_scatter('s2', 'low', 0.995, "🟢 Long", 'green', "up")

# Разделяем по типам касания
cross_r2 = (df['high'] >= df['upband2']) & (df['low'] <= df['upband2'])
above_r2 = df['low'] > df['upband2']
cross_s2 = (df['high'] >= df['loband2']) & (df['low'] <= df['loband2'])
below_s2 = df['high'] < df['loband2']

df['touch_type'] = 'none'
df.loc[cross_r2, 'touch_type'] = 'cross_r2'
df.loc[above_r2, 'touch_type'] = 'above_r2'
df.loc[cross_s2, 'touch_type'] = 'cross_s2'
df.loc[below_s2, 'touch_type'] = 'below_s2'

price_log_scale_value="log"
price_linear_scale_value="linear"
price_log_scale_title="Price (log scale)"
price_linear_scale_title="Price (linear scale)"

fig_main.update_layout(
    title="OHLC with Volume Explosion After Valley",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=3000,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.075,
            y=1.08,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(
                color="red",
                size=12,
                family="Arial"
            ),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

stats = analyze_and_print_statistics(df)
plot_all_target_candles_and_save_htmls_to_disc(df, 250)
# fig_main.show()